# 04 - Modeling
In this notebook, I'll compare different models to see which ones identify the most laundering activity while staying within a limit of 50 reviews per day.

I'll use the 11 features selected in Notebook 03. Historical data will be used to train and select the models, while the later deployment period will be used for testing.

In [14]:

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/INFO320_AML")
FILE = PROJECT / "Data" / "Processed" / "sender_day_features.parquet"

df = pd.read_parquet(FILE)

print("Total sender-days:", len(df))
print("\nSender-days by period:")
print(df["Period"].value_counts())

print("\nLaundering sender-days:", df["label"].sum())

display(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total sender-days: 3683221

Sender-days by period:
Period
Deployment    2724979
History        958242
Name: count, dtype: int64

Laundering sender-days: 9064


,sender_id,Date,payments_today,receivers_today,any_cross_border,cross_border_share_today,any_mismatch,any_high_risk,any_cash,max_amount_ratio,total_amount_ratio_today,label,Laundering_type,received_prior_7d,payments_prior_7d,payment_spike,max_receiver_fanin,Period,Sender_account
0,0,2022-10-07,1,1,False,0.0,False,False,True,0.242297,0.242297,0,NaN,0.0,0.0,1.000,1,History,8724731955
1,0,2022-11-16,1,1,False,0.0,False,False,True,0.780356,0.780356,0,NaN,0.0,0.0,1.000,1,History,8724731955
2,0,2023-01-05,1,1,False,0.0,False,False,True,0.527244,0.527244,0,NaN,0.0,0.0,1.000,1,Deployment,8724731955
3,0,2023-01-20,1,1,False,0.0,False,False,True,0.810274,0.810274,0,NaN,0.0,0.0,1.000,1,Deployment,8724731955
4,0,2023-01-25,1,1,False,0.0,False,False,True,0.132670,0.132670,0,NaN,0.0,1.0,0.875,1,Deployment,8724731955


Im using the 11 features I selected in Notebook 03. Stage A will use 9 behavioral features and Stage B will add the 2 network features to see if they improve detections

I'll use label as the target variable where 1 means laundering and 0 means normal activity.

In [15]:

BEHAVIOR = [
    "payments_today",
    "receivers_today",
    "payments_prior_7d",
    "payment_spike",
    "any_cross_border",
    "any_mismatch",
    "any_high_risk",
    "any_cash",
    "max_amount_ratio"]

NETWORK = ["max_receiver_fanin", "received_prior_7d"]

STAGE_A = BEHAVIOR
STAGE_B = BEHAVIOR + NETWORK

print("Stage A features:", len(STAGE_A))
print("Stage B features:", len(STAGE_B))
print("Missing feature columns:", set(STAGE_B) - set(df.columns))


Stage A features: 9
Stage B features: 11
Missing feature columns: set()


I'm splitting the data by date so that the models aren't trained on future laundering labels.

I'll use October and November 2022 for training and rule calibration, December for validation, and January through August 2023 for testing.

I'll keep the test period separate until the models are selected so I can evaluate how well they perform on new data.

In [16]:

df["Date"] = pd.to_datetime(df["Date"])

train = df[df["Date"] < "2022-12-01"].copy()
val = df[(df["Date"] >= "2022-12-01") & (df["Date"] < "2023-01-01")].copy()
test = df[df["Date"] >= "2023-01-01"].copy()

for name, data in [("Train", train), ("Validation", val), ("Test", test)]:
    print(name, "| Sender-days:", len(data), "| Laundering:", data["label"].sum())


Train | Sender-days: 612101 | Laundering: 1376
Validation | Sender-days: 346141 | Laundering: 827
Test | Sender-days: 2724979 | Laundering: 6861


The data was split into 612,101 training sender-days, 346,141 validation sender-days, and 2,724,979 test sender-days.

Laundering was rare across all three periods, accounting for about 0.23%–0.25% of sender-days.

Although the training data contains 1,376 laundering sender-days, the bank doesn't know which ones are laundering until investigators review them. Since the bank can only review 50 sender-days each day, my models will be trained using only those reviewed cases, not the entire dataset.

I'm creating one function to evaluate the rules and machine-learning models using the same 50-review daily limit.

The function will rank sender-days by their scores and calculate how many laundering cases were caught, along with recall and precision. This will let me compare each system fairly.

In [17]:

def evaluate(scored, score_cols, capacity=50):
    ranked = scored.sort_values(
        ["Date"] + score_cols + ["sender_id"],
        ascending=[True] + [False] * len(score_cols) + [True])

    flagged = ranked.groupby("Date").head(capacity)
    caught = flagged["label"].sum()

    return {
        "caught": caught,
        "recall": caught / scored["label"].sum(),
        "precision": caught / len(flagged)}


Before modeling, I'm checking that the selected features have no missing values, that each sender-day appears only once, and that all features are numeric or Boolean.

I already completed the main data cleaning in Notebook 02, so this is just a final check before training.

In [18]:

print("Missing values:", df[STAGE_B + ["label"]].isna().sum().sum())

print("Duplicate sender-days:", df.duplicated(["sender_id", "Date"]).sum())

print("Non-numeric features:",
      df[STAGE_B].select_dtypes(exclude=["number", "bool"]).columns.tolist())

print("Label values:", sorted(df["label"].unique()))


Missing values: 0
Duplicate sender-days: 0
Non-numeric features: []
Label values: [np.int8(0), np.int8(1)]


# Part B - Rules Based Baselines

I will now create a rules based system to compare against my machine learning models. The rules will look for unusual payment activity, large amounts, cash transactions, and international risk indicators.

Each rule that is triggered will add one point to the sender days risk score. Sender daays with higher scores will be reviewed first.

I'm using October and November data to calculate the 95th, 99th, and 99.5th percentile thresholds.

I'll compare them to see which catches the most laundering cases with a limit of 50 reviews per day. I'm using the historical labels to make the rules a stronger benchmark, while keeping December and later data separate.

In [19]:

RULE_FEATURES = ["payments_today", "receivers_today", "max_amount_ratio"]

thresholds = train[RULE_FEATURES].quantile([0.95, 0.99, 0.995])
thresholds.index = ["95th", "99th", "99.5th"]

display(thresholds)


,payments_today,receivers_today,max_amount_ratio
95th,12.0,3.0,4.342701
99th,12.0,4.0,13.854017
99.5th,13.0,5.0,35.077593


The 95th and 99th percentiles both had a limit of 12 payments per day and the limits for receivers and payment amounts increased at each level.

Higher limits mean fewer sender-days will be flagged. Next, I'll test which level catches the most laundering cases when the bank can only review 50 sender-days per day.

I'm giving each sender-day one point for every rule it triggers. The higher the score, the higher its priority for review.

I'll compare the three threshold levels to see which catches the most laundering cases with a limit of 50 reviews per day.

In [20]:

def score_rules(data, level):
    t = thresholds.loc[level]
    scored = data.copy()

    scored["rule_score"] = (
        (data["payments_today"] >= t["payments_today"]).astype(int)
        + (data["receivers_today"] >= t["receivers_today"]).astype(int)
        + (data["any_cash"] & (data["max_amount_ratio"] >= t["max_amount_ratio"])).astype(int)
        + data["any_high_risk"].astype(int)
        + (data["max_amount_ratio"] >= t["max_amount_ratio"]).astype(int)
        + (data["any_cross_border"] | data["any_mismatch"]).astype(int))

    return scored

results = []

for level in thresholds.index:
    r = evaluate(score_rules(train, level), ["rule_score", "max_amount_ratio"])
    results.append([level, int(r["caught"]), r["recall"] * 100, r["precision"] * 100])

results = pd.DataFrame(
    results, columns=["Threshold", "Caught", "Recall (%)", "Precision (%)"])

display(results.round(2))


,Threshold,Caught,Recall (%),Precision (%)
0,95th,11,0.80,0.40
1,99th,12,0.87,0.44
2,99.5th,7,0.51,0.25


The 95th percentile rules caught 11 laundering sender-days, the 99th caught 12, and the 99.5th caught 7.

The 99th percentile performed the best, catching 0.87% of the 1,376 laundering sender-days, so I'll use it for the original rules.

The results were much lower than I expected. This could potentially be a problem for training my models because investigators may find very few laundering cases to learn from.

Below im going to check how many sender days received each rule score and how many laundering cases were found at each score to better understand the results.

I'm going to use October–November data to calculate the lift of each rule, which shows how much more likely laundering is when that rule is triggered.

I'll remove rules with a lift below 1 or fewer than 100 matches, then give the remaining rules more weight based on their lift.

In [27]:

def rule_flags(data, level):
    t = thresholds.loc[level]

    return pd.DataFrame({
        "high_velocity": data["payments_today"] >= t["payments_today"],
        "high_fanout": data["receivers_today"] >= t["receivers_today"],
        "cash_large_amount": data["any_cash"] & (data["max_amount_ratio"] >= t["max_amount_ratio"]),
        "high_risk_country": data["any_high_risk"],
        "unusual_amount": data["max_amount_ratio"] >= t["max_amount_ratio"],
        "international_risk": data["any_cross_border"] | data["any_mismatch"]}, index=data.index)


flags = rule_flags(train, "99th")

fires = flags.sum()
positives = flags[train["label"] == 1].sum()
lift = ((positives / fires) / train["label"].mean()).fillna(0)

summary = pd.DataFrame({
    "Fires/day": (fires / train["Date"].nunique()).round(1),
    "Laundering": positives,
    "Lift": lift.round(3),
    "Keep": (lift >= 1) & (fires >= 100)})

display(summary)


,Fires/day,Laundering,Lift,Keep
high_velocity,624.5,0,0.000,False
high_fanout,205.3,45,1.772,True
cash_large_amount,4.6,2,3.503,True
high_risk_country,163.2,90,4.460,True
unusual_amount,111.3,61,4.432,True
international_risk,1249.4,505,3.269,True


High velocity triggered the most often but didn't identify any laundering cases, so I'll remove it.

The other five rules all had a lift above 1 and triggered at least 100 times during training. High-risk countries and unusual amounts had the highest lift, meaning laundering was over four times as common when those rules triggered.

I'll keep these five rules and use their lift to calculate weighted risk scores.

Now I'll test the rules at the 95th, 99th, and 99.5th percentiles.

For each threshold, I'll remove rules that don't meet my criteria, weight the remaining rules by their lift, and see how many laundering cases they catch with 50 reviews per day.

In [31]:

rule_weights = {}
results = []

for level in thresholds.index:
    flags = rule_flags(train, level)
    fires = flags.sum()
    lift = (flags[train["label"] == 1].sum() / fires / train["label"].mean()).fillna(0)

    rule_weights[level] = lift.where((lift >= 1) & (fires >= 100), 0)

    scored = train.copy()
    scored["rule_score"] = flags.mul(rule_weights[level]).sum(axis=1)

    caught = evaluate(scored, ["rule_score", "max_amount_ratio"])["caught"]
    results.append([level, int(caught)])

calibration = pd.DataFrame(results, columns=["Rule Threshold", "Caught"])

calibration["Recall (%)"] = (calibration["Caught"] / train["label"].sum() * 100).round(2)
calibration["Precision (%)"] = (calibration["Caught"] / (50 * train["Date"].nunique()) * 100).round(2)

display(calibration.style.hide(axis="index"))


Rule Threshold,Caught,Recall (%),Precision (%)
95th,26,1.890000,0.950000
99th,31,2.250000,1.130000
99.5th,25,1.820000,0.910000


The 99th percentile performed the best, catching 31 laundering sender-days, compared to 26 at the 95th precentile and 25 at the 99.5th precentile.

the 99th also caught 2.25% of all laundering sender-days while reviewing only 50 per day.

I'll use the 99th percentile for my rules and test how well they work on later months.

Now I'll create a function that uses the selected thresholds and weights to calculate a risk score for each sender day. This will let me apply the same rules to the historical reviews and later testing.

In [32]:

best_level = "99th"

def score_rules_recalibrated(data):
    scored = data.copy()
    flags = rule_flags(data, best_level)

    scored["rule_score"] = flags.mul(rule_weights[best_level]).sum(axis=1)

    return scored

print("Selected setting:", best_level)
print("Final rule weights:")
print(rule_weights[best_level].round(3))


Selected setting: 99th
Final rule weights:
high_velocity         0.000
high_fanout           1.772
cash_large_amount     3.503
high_risk_country     4.460
unusual_amount        4.432
international_risk    3.269
dtype: float64


# Part C - Simulated Historical Investigations

Now i will simulate what investigators would actually review.

Each day, the bank will select the 45 highest-ranked cases, and I'll randomly select another 5 from ranks 51–200.

Only reviewed cases will be used to train the models, since the bank wouldn't know the outcomes of cases it hasn't investigated.

In [33]:

history = pd.concat([train, val], ignore_index=True)
history = score_rules_recalibrated(history)

ranked = history.sort_values(
    ["Date", "rule_score", "max_amount_ratio", "sender_id"],
    ascending=[True, False, False, True]).copy()

ranked["rank"] = ranked.groupby("Date").cumcount() + 1

rules45 = ranked[ranked["rank"] <= 45].copy()
random_pool = ranked[ranked["rank"].between(51, 200)]
random5 = random_pool.groupby("Date").sample(n=5, random_state=42).copy()

rules45["review_source"] = "Rules"
random5["review_source"] = "Random"

reviewed = pd.concat([rules45, random5])

print("Total reviewed:", len(reviewed))
print("Rule-selected:", len(rules45))
print("Randomly selected:", len(random5))
print("Days reviewed:", reviewed["Date"].nunique())


Total reviewed: 4300
Rule-selected: 3870
Randomly selected: 430
Days reviewed: 86


The bank reviewed 4,300 sender-days across 86 days. Of those, 3,870 were selected by the rules and 430 were randomly selected.

These are the only cases whose labels the models will be allowed to learn from.

Im going to check how many laundering cases were found during the reviews.

I'll compare the cases found by the rules and random selection, and separate the training and validation results to see how many examples the models will have to learn from.

In [34]:

reviewed["Split"] = "Validation"
reviewed.loc[reviewed["Date"] < "2022-12-01", "Split"] = "Training"

summary = reviewed.groupby(["Split", "review_source"])["label"].agg(
    Reviewed="size",
    Laundering="sum")

display(summary)

print("Total laundering found:", reviewed["label"].sum())



Reviewed  Laundering
Split      review_source                      
Training   Random              275           2
           Rules              2475          27
Validation Random              155           2
           Rules              1395          23

Total laundering found: 54


Of the 4,300 sender days reviewed, 54 laundering sender days were found. 29 were found during training and 25 during validation.

This is a concern because my models would only have 29 laundering examples to learn from. The rules missed most laundering activity, leaving very little useful training data.

I'll keep these results as part of my original experiment, but I may need to test a separate model with more labeled data to see how much this affects detection.

I'm saving the 4,300 reviewed sender-days so I can reuse them when training the models and testing the feedback loop later.

In [35]:

reviewed.to_parquet(
    PROJECT / "Data" / "Processed" / "historical_reviews.parquet",
    index=False)

print("Reviewed cases saved:", len(reviewed))

Reviewed cases saved: 4300


# Part D - Machine Learning Models

Now I'll train logistic regression and LightGBM using the cases investigators reviewed from Oct and Nov and then compare their preformance on Dec.

After that, I'll retrain both models using all the reviewed cases from October through December before testing them on January–August data.

In [36]:

train_reviewed = reviewed[reviewed["Date"] < "2022-12-01"].copy()
val_reviewed = reviewed[reviewed["Date"] >= "2022-12-01"].copy()

print("Training cases:", len(train_reviewed))
print("Training laundering cases:", train_reviewed["label"].sum())

print("Validation cases:", len(val_reviewed))
print("Validation laundering cases:", val_reviewed["label"].sum())

if val_reviewed["label"].sum() < 30:
    print("Model selection: Logistic Regression (fallback)")


Training cases: 2750
Training laundering cases: 29
Validation cases: 1550
Validation laundering cases: 25
Model selection: Logistic Regression (fallback)


The results confirm I have 29 laundering cases for training and 25 for validation and since December has fewer than 30 positive cases, I'll have to use logistic regression as my selected model under the fallback rule because its simpler and easier to interpret, but I will still compare both models.

Now I will train the first ML model which will be the logistic regression model using the 9 behavior features. Since laundering cases are rare, I'll give them more weight during training. I'll also adjust the feature values so large numbers don't have too much influence on the model.

In [37]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline

lr_model = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))

lr_model.fit(train_reviewed[STAGE_A].astype(float), train_reviewed["label"])

print("Logistic regression trained")
print("Training cases:", len(train_reviewed))
print("Training laundering cases:", train_reviewed["label"].sum())


Logistic regression trained
Training cases: 2750
Training laundering cases: 29


Now I will train the second ML model, LightGBM using the same 9 behavior features. This model uses decision trees to find more complex patterns in the data. Just like the regression model I'll also adjust for the small number of laundering cases so they have more weight during training.

In [38]:

from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=7,
    min_child_samples=20,
    class_weight="balanced",
    random_state=42,
    verbosity=-1)

lgbm_model.fit(train_reviewed[STAGE_A], train_reviewed["label"])

print("LightGBM trained")
print("Training cases:", len(train_reviewed))
print("Training laundering cases:", train_reviewed["label"].sum())

LightGBM trained
Training cases: 2750
Training laundering cases: 29


Since we learned about Random Forests in class, I also want to test one using the same 9 behavioral features.

I'll compare its performance against logistic regression and LightGBM to see how the model we learned about in class performs.

In [39]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1)

rf_model.fit(train_reviewed[STAGE_A], train_reviewed["label"])

print("Random Forest trained")
print("Training cases:", len(train_reviewed))
print("Training laundering cases:", train_reviewed["label"].sum())


Random Forest trained
Training cases: 2750
Training laundering cases: 29


Now I'll test all three models on the reviewed December cases to see how well they identify laundering.

I'll use average precision to compare their performance.

In [40]:

from sklearn.metrics import average_precision_score

models = {
    "Logistic Regression": lr_model,
    "LightGBM": lgbm_model,
    "Random Forest": rf_model}

results = []

for name, model in models.items():
    scores = model.predict_proba(val_reviewed[STAGE_A])[:, 1]
    ap = average_precision_score(val_reviewed["label"], scores)
    results.append([name, ap])

comparison = pd.DataFrame(results, columns=["Model", "Average Precision"])
display(comparison.round(4).style.hide(axis="index"))


Model,Average Precision
Logistic Regression,0.096500
LightGBM,0.105800
Random Forest,0.062100


LightGBM performed the best with an average precision of 0.1058, followed closely by logistic regression at 0.0965. Random Forest performed the worst at 0.0621.

My original selection rule was to choose logistic regression if December had fewer than 30 laundering cases or the difference in average precision was less than 0.01. Both conditions were met, with 25 laundering cases and a difference of 0.0093.

Although the rule points to logistic regression, I beleive the results are too close to confidently say which model is better. So I'll keep both models for further testing and set Random Forest aside.

Now im going to retrain the logistic regression and LightGBM models using all the reviewed cases from October through December. This will give both models more laundering examples to learn from before testing them on January–August.

In [41]:

from sklearn.base import clone

lr_final = clone(lr_model)
lgbm_final = clone(lgbm_model)

lr_final.fit(reviewed[STAGE_A].astype(float), reviewed["label"])
lgbm_final.fit(reviewed[STAGE_A], reviewed["label"])

print("Both models retrained")
print("Training cases:", len(reviewed))
print("Laundering cases:", reviewed["label"].sum())


Both models retrained
Training cases: 4300
Laundering cases: 54


# Part E - Deployment Evaluation

Ill now test how well the rules and Ml models identify laundering from January through August.

I will compare the rules, logistic regression, and LightGBM and i will also include a baseline that ranks cases by paymont amount alone

Each model will have the same limit of 50 reviews per day so I can compare how many laundering cases they catch.

I'll use each system to score the January–August sender days. The scores will be used to rank cases for investigation, with higher scores receiving priority.

In [42]:

deployment = test[["Date", "sender_id", "label", "max_amount_ratio"]].copy()

deployment["rules"] = score_rules_recalibrated(test)["rule_score"].to_numpy()
deployment["amount_only"] = test["max_amount_ratio"].to_numpy()

deployment["logistic_regression"] = lr_final.predict_proba(test[STAGE_A].astype(float))[:, 1]
deployment["lightgbm"] = lgbm_final.predict_proba(test[STAGE_A])[:, 1]

print("Sender-days scored:", len(deployment))
print("Days:", deployment["Date"].nunique())
print("Laundering sender-days:", deployment["label"].sum())


Sender-days scored: 2724979
Days: 235
Laundering sender-days: 6861


Now I'll compare how well all three of the models perform with a limit of 50 reviews per day.

I'll measure how many laundering sender days each system catches, along with recall and precision, to see whether the ML models perform better than the rules.

In [43]:

systems = {
    "Rules": ["rules", "max_amount_ratio"],
    "Amount Only": ["amount_only"],
    "Logistic Regression": ["logistic_regression", "max_amount_ratio"],
    "LightGBM": ["lightgbm", "max_amount_ratio"]}

results = []

for name, columns in systems.items():
    r = evaluate(deployment, columns)
    results.append([
        name,
        int(r["caught"]),
        round(r["recall"] * 100, 2),
        round(r["precision"] * 100, 2)])

performance = pd.DataFrame(
    results,
    columns=["System", "Caught", "Recall (%)", "Precision (%)"])

display(performance.style.hide(axis="index"))


System,Caught,Recall (%),Precision (%)
Rules,168,2.450000,1.430000
Amount Only,124,1.810000,1.060000
Logistic Regression,28,0.410000,0.240000
LightGBM,228,3.320000,1.940000


LightGBM performed the best, catching 228 laundering sender-days compared to 168 for the rules. That's about a 36% improvement with the same limit of 50 reviews per day.

Very suprisingly logistic regression performed much worse, catching only 28 cases. It even performed worse than using payment amounts alone, which caught 124.

This was surprising because logistic regression and LightGBM had similar results during validation. It shows that their December results didn't necessarily reflect how well they would perform on later data.

Now I'll use bootstrapping which is where I randomly resample the data multiple times to see how much the results might change to see how consistent the differences are between the ML models and the rules.

 I'll resample the 235 days 1,000 times and calculate a 95% confidence interval to see whether the models consistently perform better or worse than the rules.

In [44]:

daily = pd.DataFrame()

for name, col in [("Rules", "rules"),
                  ("Logistic Regression", "logistic_regression"),
                  ("LightGBM", "lightgbm")]:

    top50 = deployment.sort_values(
        ["Date", col, "max_amount_ratio", "sender_id"],
        ascending=[True, False, False, True]).groupby("Date").head(50)

    daily[name] = top50.groupby("Date")["label"].sum()


rng = np.random.default_rng(42)

for model in ["Logistic Regression", "LightGBM"]:
    diff = (daily[model] - daily["Rules"]).to_numpy()
    samples = rng.choice(diff, size=(1000, len(diff))).sum(axis=1)
    low, high = np.percentile(samples, [2.5, 97.5])

    print(f"{model}: {diff.sum():+d} cases")
    print(f"95% CI: [{low:.0f}, {high:.0f}]")


Logistic Regression: -140 cases
95% CI: [-168, -112]
LightGBM: +60 cases
95% CI: [27, 92]


The bootstrap confirmed the results and shows that LightGBM caught 60 more laundering cases than the rules, while logistic regression caught 140 fewer.

Both confidence intervals excluded zero, which suggests these differences were consistent across the test days. This now confirms it and gives me more confidence that LightGBM performed better than the rules.

I'm going to calculate the most laundering cases the bank could possibly catch with 50 reviews per day, assuming it always selected the right cases. This will give me a maximum to compare my models against.

In [45]:

daily_positives = deployment.groupby("Date")["label"].sum()

max_caught = daily_positives.clip(upper=50).sum()
total_positives = deployment["label"].sum()
total_reviews = 50 * len(daily_positives)

print("Maximum cases caught:", max_caught)
print("Maximum recall:", round(max_caught / total_positives * 100, 2), "%")
print("Maximum precision:", round(max_caught / total_reviews * 100, 2), "%")


Maximum cases caught: 6858
Maximum recall: 99.96 %
Maximum precision: 58.37 %


The results show that the bank could theoretically catch 6,858 of the 6,861 laundering sender days with only 50 reviews per day.

This means the review limit isn't really the main problem. The bigger challenge is correctly ranking the cases so that investigators review the ones most likely to involve laundering.

Im going to train both ML models using all the October–December sender days, rather than only the reviewed cases.

This will let me see if giving the models more labeled training data improves their ability to detect laundering.

In [46]:

history_full = pd.concat([train, val], ignore_index=True)

lr_full = clone(lr_model)
lgbm_full = clone(lgbm_model)

lr_full.fit(history_full[STAGE_A].astype(float), history_full["label"])
lgbm_full.fit(history_full[STAGE_A], history_full["label"])

print("Full-label models trained")
print("Training cases:", len(history_full))
print("Laundering cases:", history_full["label"].sum())


Full-label models trained
Training cases: 958242
Laundering cases: 2203


And now I will test both ML models on the January–August data using the same limit of 50 reviews per day to compare the results to see whether training with more labeled cases improved detection.

In [47]:

deployment["full_lr"] = lr_full.predict_proba(test[STAGE_A].astype(float))[:, 1]
deployment["full_lgbm"] = lgbm_full.predict_proba(test[STAGE_A])[:, 1]

results = []

for name, column in [
    ("Logistic Regression", "full_lr"),
    ("LightGBM", "full_lgbm")]:
    r = evaluate(deployment, [column, "max_amount_ratio"])
    results.append([
        name,
        int(r["caught"]),
        round(r["recall"] * 100, 2),
        round(r["precision"] * 100, 2)])

full_performance = pd.DataFrame(
    results,
    columns=["Model", "Caught", "Recall (%)", "Precision (%)"])

display(full_performance.style.hide(axis="index"))


Model,Caught,Recall (%),Precision (%)
Logistic Regression,205,2.990000,1.740000
LightGBM,429,6.250000,3.650000


Both models performed much better when trained using all the historical labels. Logistic regression increased from 28 to 205 laundering cases, while LightGBM increased from 228 to 429.

This suggests that the limited number of laundering examples found during investigations is holding back the models, but its not the only thing holding them back because even with all the training labels, LightGBM only caught 6.25% of laundering cases, so there's still a lot of room for improvement.

AND interestignly the regression model with all the historical labels still under preformed LightGBM when it was only trained on 54 labels  

# Part F - Breakdowns

In this section I'll look more closely at the results to see how detection changes by month, laundering type, and number of daily reviews. I'll also estimate how much investigator time could be saved.

I will compare how many laundering cases each system caught from January through August and calculate monthly recall to see how detection performance changes over time.

In [48]:

monthly = daily.copy()
monthly["Month"] = monthly.index.to_period("M")
monthly = monthly.groupby("Month").sum()

monthly["Total Laundering"] = deployment.groupby(
    deployment["Date"].dt.to_period("M"))["label"].sum()

for model in ["Rules", "Logistic Regression", "LightGBM"]:
    monthly[model + " Recall (%)"] = (
        monthly[model] / monthly["Total Laundering"] * 100).round(2)

display(monthly)


,Rules,Logistic Regression,LightGBM,Total Laundering,Rules Recall (%),Logistic Regression Recall (%),LightGBM Recall (%)
Month,,,,,,,
2023-01,15,5,22,880,1.70,0.57,2.50
2023-02,22,4,21,888,2.48,0.45,2.36
2023-03,18,4,24,849,2.12,0.47,2.83
2023-04,22,2,43,888,2.48,0.23,4.84
2023-05,17,0,20,789,2.15,0.00,2.53
2023-06,29,6,43,913,3.18,0.66,4.71
2023-07,33,5,38,904,3.65,0.55,4.20
2023-08,12,2,17,750,1.60,0.27,2.27


LightGBM performed the best in 7 of the 8 months and regression struggled throughout the test period.

Detection rates did change from month to month. LightGBM had its highest recall in April at 4.84%, but dropped to 2.27% by August.

This shows that model performance can change over time, which is why monitoring the models after deployment is important.

Im going to compare how well each model detected the specific different types of laundering in the dataset.

'll calculate how many cases of each type were caught and their recall to see which laundering schemes are easier or harder to identify.

In [49]:

deployment["Laundering_type"] = test["Laundering_type"].astype(str).to_numpy()

typology = (
    deployment[deployment["label"] == 1]["Laundering_type"]
    .value_counts()
    .rename_axis("Laundering Type")
    .reset_index(name="Total Laundering"))

for name, col in [
    ("Rules", "rules"),
    ("Logistic Regression", "logistic_regression"),
    ("LightGBM", "lightgbm")]:
    top50 = deployment.sort_values(
        ["Date", col, "max_amount_ratio", "sender_id"],
        ascending=[True, False, False, True]).groupby("Date").head(50)

    caught = top50[top50["label"] == 1]["Laundering_type"].value_counts()

    typology[name] = typology["Laundering Type"].map(caught).fillna(0).astype(int)
    typology[name + " Recall (%)"] = (
        typology[name] / typology["Total Laundering"] * 100).round(2)

display(typology)


,Laundering Type,Total Laundering,Rules,Rules Recall (%),Logistic Regression,Logistic Regression Recall (%),LightGBM,LightGBM Recall (%)
0,structuring,1373,2,0.15,15,1.09,2,0.15
1,cash_withdrawal,801,1,0.12,0,0.00,2,0.25
2,deposit_send,732,25,3.42,0,0.00,2,0.27
3,smurfing,710,2,0.28,0,0.00,1,0.14
4,layered_fan_in,528,7,1.33,0,0.00,2,0.38
5,layered_fan_out,322,1,0.31,3,0.93,0,0.00
6,stacked_bipartite,322,0,0.00,0,0.00,0,0.00
7,cycle,312,14,4.49,2,0.64,0,0.00
8,scatter_gather,264,3,1.14,1,0.38,1,0.38
9,bipartite,257,5,1.95,1,0.39,0,0.00


These results were interesting LightGBM performed very well at detecting the two behavioral change types, catching around half of each.

However, 211 of the 228 cases it caught came from just these two types of laundering and it struggled with most of the other laundering types, especially those involving networks of accounts.

This suggests that more advanced network features could help improve detection which we will test in 05 Network Features and Feedback Loops.

Now ill test wheather increasing the review capacity to see if the number of reviews daily makes a difference

Ill compare 25, 50, and 100 reviews per day.

In [50]:

results = []

for capacity in [25, 50, 100]:
    for name in ["Rules", "Logistic Regression", "LightGBM"]:
        r = evaluate(deployment, systems[name], capacity=capacity)

        results.append([
            capacity, name, int(r["caught"]),
            round(r["recall"] * 100, 2),
            round(r["precision"] * 100, 2)])

capacity_results = pd.DataFrame(
    results,
    columns=["Reviews/Day", "System", "Caught", "Recall (%)", "Precision (%)"])

display(capacity_results.style.hide(axis="index"))


Reviews/Day,System,Caught,Recall (%),Precision (%)
25,Rules,81,1.180000,1.380000
25,Logistic Regression,28,0.410000,0.480000
25,LightGBM,156,2.270000,2.660000
50,Rules,168,2.450000,1.430000
50,Logistic Regression,28,0.410000,0.240000
50,LightGBM,228,3.320000,1.940000
100,Rules,373,5.440000,1.590000
100,Logistic Regression,30,0.440000,0.130000
100,LightGBM,311,4.530000,1.320000


This provides another interesting result LightGBM performed better than the rules at 25 and 50 reviews per day. However, when I increased the limit to 100, the rules caught 373 laundering cases compared to LightGBM's 311 This shows that LightGBM is better at prioritizing cases when investigators have limited time but gets outpreformed when the review limit is increased.

Logistic regression barely improved as the number of reviews increased.

To test for investigator efficiency I'll find the smallest number of daily reviews LightGBM needs to catch at least as many laundering cases as the rules catch with 50 reviews per day by checking every capacity from 1 to 50 to find the first one where LightGBM matches or exceeds the rules' 168 detections.

In [51]:

rules_caught = 168

ranked = deployment.sort_values(
    ["Date", "lightgbm", "max_amount_ratio", "sender_id"],
    ascending=[True, False, False, True]).groupby("Date").head(50).copy()

ranked["rank"] = ranked.groupby("Date").cumcount() + 1

catches = ranked.groupby("rank")["label"].sum().cumsum()

minimum = catches[catches >= rules_caught].index.min()
reviews_saved = (50 - minimum) * 235

print("Rules at 50/day:", rules_caught)
print("LightGBM minimum reviews/day:", minimum)
print("LightGBM cases caught:", catches.loc[minimum])
print("Reviews saved:", reviews_saved)


Rules at 50/day: 168
LightGBM minimum reviews/day: 28
LightGBM cases caught: 170
Reviews saved: 5170


LightGBM was able to catch 170 laundering cases with only 28 reviews per day, compared to the rules model catching 168 cases at 50 reviews per day. This means 5,170 fewer reviews were needed.

I'll estimate how much investigator time this potentially save using 37 minutes per review, based on a 2025 Boston Consulting Group banking compliance study.

Source: BCG — Risky Times and Cost Pressure Call for Innovation in Bank Compliance, Exhibit 3

In [52]:

minutes_per_review = 37
hours_saved = reviews_saved * minutes_per_review / 60

print("Reviews saved:", reviews_saved)
print("Assumed minutes per review:", minutes_per_review)
print("Estimated investigator hours saved:", round(hours_saved))


Reviews saved: 5170
Assumed minutes per review: 37
Estimated investigator hours saved: 3188


Over the 235 days, this could save 5,170 reviews, or an estimated 3,188 investigator hours.

# Part G - Findings and outputs

I'll summarize the main findings from my analysis and save the results and models for Tableau and the next notebook.

## Main Findings



*   Model performance: LightGBM performed the best, catching 228 laundering sender-days compared to 168 for the rules and only 28 for logistic regression. My original selection rule favored logistic regression but initial results were to close to let one modle go so i selected and tested both

* Statistical results: The bootstrap results supported these differences. LightGBM caught 60 more cases than the rules, with a 95% confidence interval of +27 to +92.

* Laundering types: LightGBM caught 211 of its 228 cases from just two behavioral change types. It struggled with most other types, especially those involving networks of accounts.

* Investigator efficiency: LightGBM could catch about the same number of cases as the rules with 44% fewer reviews. This could save an estimated 3,188 investigator hours.

Overall, LightGBM performed the best, but it still missed most laundering cases. In Notebook 05, I'll test whether adding network features and using a feedback loop can improve detection.



## Saving Results and Models

Now I'll save my results, model scores, and reviewed cases so I can use them in Tableau and Notebook 05. I'll also save the trained models and their settings so I can reproduce my results later.

Im saving the risk scores each model gave to the sender days and marking which each case each model would have been selected for review. This will let me compare their results later without having to run the models again.

In [53]:

score_columns = {
    "rules": ["rules", "max_amount_ratio"],
    "amount_only": ["amount_only"],
    "logistic_regression": ["logistic_regression", "max_amount_ratio"],
    "lightgbm": ["lightgbm", "max_amount_ratio"],
    "full_lr": ["full_lr", "max_amount_ratio"],
    "full_lgbm": ["full_lgbm", "max_amount_ratio"]}

for name, columns in score_columns.items():
    ranked = deployment.sort_values(
        ["Date"] + columns + ["sender_id"],
        ascending=[True] + [False] * len(columns) + [True])

    selected = ranked.groupby("Date").head(50).index
    deployment[name + "_top50"] = deployment.index.isin(selected)

output = PROJECT / "Data" / "Processed"
output.mkdir(parents=True, exist_ok=True)

deployment.to_parquet(output / "deployment_scores.parquet", index=False)

print("Deployment sender-days saved:", len(deployment))
print("Systems saved:", len(score_columns))
print("File: deployment_scores.parquet")


Deployment sender-days saved: 2724979
Systems saved: 6
File: deployment_scores.parquet


Im saving the tables showing monthly performance, laundering types, and review capacity, along with the reviewed cases used for training. This will let me use the results in Tableau and Notebook 05 without having to recalculate them.

In [54]:

monthly_export = monthly.reset_index()
monthly_export["Month"] = monthly_export["Month"].astype(str)

tables = {
    "model_performance.csv": performance,
    "full_label_performance.csv": full_performance,
    "monthly_performance.csv": monthly_export,
    "typology_performance.csv": typology,
    "capacity_performance.csv": capacity_results}

for filename, table in tables.items():
    table.to_csv(output / filename, index=False)

reviewed.to_csv(output / "historical_reviews.csv", index=False)

efficiency = pd.DataFrame([{
    "Rules Reviews/Day": 50,
    "Rules Caught": rules_caught,
    "LightGBM Reviews/Day": minimum,
    "LightGBM Caught": catches.loc[minimum],
    "Reviews Saved": reviews_saved,
    "Minutes/Review": minutes_per_review,
    "Estimated Hours Saved": round(hours_saved)}])

efficiency.to_csv(output / "investigator_efficiency.csv", index=False)

print("Results tables saved:", len(tables))
print("Reviewed cases saved:", len(reviewed))
print("Efficiency results saved")


Results tables saved: 5
Reviewed cases saved: 4300
Efficiency results saved


Im savingthe trained models and the settings I used, including the features and rule weights. This will let me use the same models later without having to train them again.

In [55]:

import joblib
import json

models_saved = {
    "logistic_regression": lr_final,
    "lightgbm": lgbm_final,
    "random_forest_validation": rf_model,
    "full_label_logistic_regression": lr_full,
    "full_label_lightgbm": lgbm_full}

joblib.dump(models_saved, output / "trained_models.joblib")

settings = {
    "features": STAGE_A,
    "rule_setting": best_level,
    "rule_thresholds": thresholds.loc[best_level].to_dict(),
    "rule_weights": rule_weights[best_level].to_dict(),
    "review_capacity": 50,
    "review_policy": "Top 45 rules plus 5 random from ranks 51-200",
    "training_period": "October-December 2022",
    "testing_period": "January-August 2023",
    "random_seed": 42,
    "logistic_regression": lr_final.named_steps["logisticregression"].get_params(),
    "lightgbm": lgbm_final.get_params(),
    "random_forest": rf_model.get_params()}

with open(output / "model_settings.json", "w") as f:
    json.dump(settings, f, indent=4, default=str)

print("Models saved:", len(models_saved))
print("Settings saved: model_settings.json")


Models saved: 5
Settings saved: model_settings.json
